# GPT: Scale, Analysis, and ChatGPT

## Project
Study GPT as a decoder-only architecture and a training procedure. An experiment illustrates the role of the causal mask; we estimate parameter counts and attention cost, then compare GPT-3, GPT-4, and ChatGPT. GPT-4 details are not public enough to reconstruct its parameter count.

## Theoretical Framework: What Exactly Are We Scaling in GPT?

### The base model and prediction objective
GPT is an autoregressive probabilistic model with factorization $p_\theta(x_{1:T})=\prod_t p_\theta(x_t\mid x_{<t})$. A transformer block updates each token representation through causal self-attention and an MLP. Pretraining optimizes the negative log-likelihood of text. The architecture defines a function family, while data, optimizer, learning-rate schedule, and training objective influence which functions are learned.

### Computation cost versus parameter count
For a layer of width $d$, the Q, K, and V projections each have $d^2$ parameters, the output projection has $d^2$, and an MLP $d\to4d\to d$ has $4d^2+4d^2$. This gives approximately $12d^2$ weights per block, excluding biases and normalization. This is not the parameter count of an entire model: embeddings, all layers, normalization, and any shared weights must also be included.

Attention on a sequence of length T costs $O(T^2d)$, while projections and the MLP cost about $O(Td^2)$. The quadratic relationship is over pairs of positions, not necessarily the entire model budget at every T. Memory-efficient algorithms can reduce storage of the attention matrix, but do not automatically remove all compute costs or change model quality.

### Why a larger model does not automatically make a better assistant
Pretraining teaches a model to predict text from a corpus, not directly to be helpful. Supervised fine-tuning (SFT) teaches instruction-response formats. In RLHF approaches, people compare responses, a reward model approximates those preferences, and policy optimization shifts the response distribution. DPO directly optimizes preferred responses relative to rejected responses, usually against a reference model. These procedures make different assumptions and do not guarantee truthfulness; preferences are not the same as factual correctness.

### The mask experiment as a methodology check
If a model can see the target token in its context during training, it may achieve low loss using a shortcut unavailable during generation. GPT therefore allows only keys j≤t at position t. The comparison with an unmasked model illustrates the information available to the computation; it is not an architecture benchmark. Results depend on the synthetic task and experiment details.

### GPT-3, GPT-4, and ChatGPT: different levels of description
GPT-3 refers to the model family described in a paper that reported a 175-billion-parameter variant and few-shot evaluations. GPT-4 is a later system whose technical report described results and limitations but did not disclose full architecture, data, or size details. ChatGPT describes a conversational assistant experience and evolving models and system components. Distinguish the architecture name, a specific model version, and a product with tools and policies. This notebook covers durable concepts; check current documentation when you need up-to-date versions or API details.

### Analyzing a model without relying on one misleading metric
Cross-entropy and perplexity evaluate the probability assigned to sequences. Benchmarks measure a particular set of tasks and are sensitive to prompt format, training data, and contamination. Assistant evaluation should combine controlled tests, error analysis, calibration, human evaluation, and operational measures such as latency, throughput, memory, and cost. Interpret a result within its specific protocol, not as a universal measure of intelligence.

## 1. Autoregression and Information Leakage
GPT models $p(x_1,\ldots,x_T)=\prod_t p(x_t\mid x_{<t})$. Pretraining minimizes the negative log-likelihood of the next token. Without a mask, the model could read future tokens. The following synthetic experiment illustrates the issue; unmasked attention is appropriate for some encoders, but not for GPT.

In [ ]:
import math,torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(4);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyLM(nn.Module):
    def __init__(self,causal=True):
        super().__init__();self.causal=causal;self.e=nn.Embedding(16,32);self.qkv=nn.Linear(32,96);self.o=nn.Linear(32,16)
    def forward(self,x):
        B,T=x.shape;q,k,v=self.qkv(self.e(x)).chunk(3,-1);s=q@k.transpose(-2,-1)/math.sqrt(32)
        if self.causal:s=s.masked_fill(~torch.tril(torch.ones(T,T,dtype=torch.bool,device=x.device)),torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
seq=torch.tensor([[i%4 for i in range(20)] for _ in range(32)],device=device);x,y=seq[:,:-1],seq[:,1:];hist={}
for name,causal in [("causal",True),("bez maski",False)]:
    model=TinyLM(causal).to(device);opt=torch.optim.AdamW(model.parameters(),lr=.01);ls=[]
    for _ in range(100):
        z=model(x);loss=F.cross_entropy(z.reshape(-1,16),y.reshape(-1));opt.zero_grad();loss.backward();opt.step();ls.append(loss.item())
    hist[name]=ls
for n,v in hist.items():plt.plot(v,label=n)
plt.yscale("log");plt.xlabel("krok");plt.ylabel("cross entropy");plt.legend();plt.grid();plt.show()

## 2. Parameters and Compute Cost
For $d=d_{model}$, QKV and output projections have about $4d^2$ parameters, while the MLP $d\to4d\to d$ has $8d^2$. Together, a block has about $12d^2$ parameters, plus biases and normalization. Attention weights require $O(T^2)$ memory, and attention computation costs $O(T^2d)$. Doubling context length increases this part of the cost by about four times.

In [ ]:
def block_params(d):return 12*d*d+13*d
for d in (64,256,768,2048):print(d,f"~{block_params(d):,} parameters/block")
T=[128,256,512,1024,2048,4096]
plt.loglog(T,[(t/128)**2 for t in T],marker="o");plt.xlabel("sequence length T");plt.ylabel("relative cost (T=128)");plt.title("Quadratic complexity of attention");plt.grid(True,which="both");plt.show()

## 3. From Base GPT to an Assistant
A typical pipeline (which varies by system) includes pretraining on next-token prediction over a corpus; supervised fine-tuning (SFT) on instruction-response examples; preference tuning such as RLHF (reward model plus policy optimization) or DPO; and product integration with prompts, tools, routing, and filters. Pretraining does not guarantee instruction following, truthfulness, or safety.

## 4. GPT-3, GPT-4, and ChatGPT

| Name | Meaning | What is known / limitations |
|---|---|---|
| GPT-3 | Language model described in 2020 | The paper described a variant with up to 175 billion parameters and few-shot capabilities. |
| GPT-4 | Later general-purpose model | The report describes evaluations but does not disclose many architecture, data, or size details. |
| ChatGPT | Conversational assistant | A product/family of dialogue systems, not a synonym for one architecture or one fixed set of weights. |

Do not carry GPT-3's parameter count over to GPT-4. ChatGPT covers evolving systems; this section is not current API documentation.

## 5. Model Analysis and Exercise
Analyze the token level (logits, entropy, temperature), sequence level (loss, perplexity), task level (quality, calibration, robustness, leakage), and system level (latency, throughput, cost, context). Perplexity measures text modeling, but does not directly measure factuality, conversational usefulness, or safety.

**Project:** prepare an experiment card for the mini-GPT from the previous notebook: dataset size, tokenizer, parameter count, train/validation loss and PPL, decoding settings, and three generations. Describe one limitation that temperature alone cannot fix.

## Summary
GPT predicts tokens autoregressively. A causal mask blocks leakage, and standard attention scales quadratically with context length. Assistants add fine-tuning and system components.